In [103]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 64
batch_size = 128
max_iters = 3000
eval_interval = 100
learning_rate = 3e-4
eval_iters = 100
n_embd = 384
n_layer = 8
n_head = 8
dropout = 0.2
num_experts = 4
top_k = 2

cuda


In [104]:
chars = ""
with open('vocab.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    chars = sorted(set(text))

vocab_size = len(chars)

In [105]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int.get(c, 0) for c in s]
decode = lambda l: ''.join([int_to_string.get(i, '') for i in l])

In [106]:
import mmap
import random
import os

def get_random_chunk(split):
    # Pool of training files (literature + code)
    train_files = ["train_split.txt"]
    if os.path.exists("code_train_split.txt"):
        train_files.append("code_train_split.txt")
        
    filename = random.choice(train_files) if split == 'train' else "train_split.txt"
    if not os.path.exists(filename):
        filename = "train_split.txt"
        
    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            file_size = len(mm)
            max_start = file_size - block_size * batch_size
            if max_start <= 0:
                mm.seek(0)
                block = mm.read()
            else:
                start_pos = random.randint(0, max_start)
                mm.seek(start_pos)
                block = mm.read(block_size * batch_size)

            decoded_block = block.decode('utf-8', errors='ignore').replace('\r', '')
            data_chunk = torch.tensor(encode(decoded_block), dtype=torch.long)
    return data_chunk

def get_batch(split):
    data_chunk = get_random_chunk(split)
    if len(data_chunk) <= block_size:
        # Fallback if chunk is too small
        data_chunk = torch.tensor(encode(text[:10000]), dtype=torch.long)
    
    ix = torch.randint(len(data_chunk) - block_size, (batch_size,))
    x = torch.stack([data_chunk[i:i+block_size] for i in ix])
    y = torch.stack([data_chunk[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs shape:', x.shape)
print('targets shape:', y.shape)

inputs shape: torch.Size([128, 64])
targets shape: torch.Size([128, 64])


In [107]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in {'train', 'val'}:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [108]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B, T, C = x.shape
        k = self.key(x) # (B, T, hs)
        q = self.query(x) # (B, T, hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # (B, T, F) -> (B, T, { h1, h1, h1, h1, h2, h2, h2, h2, h3, h3, h3, h3 })
        out = self.dropout(self.proj(out))
        return out

class Expert(nn.Module):
    """ An individual expert network (standard FFN) """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class MixtureOfExpertsFeedForward(nn.Module):
    """ Mixture of Experts FeedForward with Top-k Gating """
    def __init__(self, n_embd, num_experts=4, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.experts = nn.ModuleList([Expert(n_embd) for _ in range(num_experts)])
        self.gate = nn.Linear(n_embd, num_experts, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)
        
        gate_logits = self.gate(x_flat) # (B*T, num_experts)
        weights, selected_experts = torch.topk(F.softmax(gate_logits, dim=-1), self.top_k, dim=-1) # (B*T, top_k)
        weights = weights / weights.sum(dim=-1, keepdim=True)
        
        out = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            batch_idx, nth_expert = torch.where(selected_experts == i)
            if batch_idx.numel() == 0:
                continue
            
            tokens_for_expert = x_flat[batch_idx]
            expert_out = expert(tokens_for_expert)
            
            weight_for_expert = weights[batch_idx, nth_expert].unsqueeze(-1)
            out.index_add_(0, batch_idx, expert_out * weight_for_expert)
            
        return out.view(B, T, C)

class Block(nn.Module):
    """ Transformer block with Mixture of Experts (MoE) """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        n_exp = globals().get('num_experts', 4)
        t_k = globals().get('top_k', 2)
        self.ffwd = MixtureOfExpertsFeedForward(n_embd, num_experts=n_exp, top_k=t_k)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape

        # idx and targets are both (B, T) tensor of integers
        tok_emb = self.token_embedding_table(index) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)
        x = self.blocks(x) # (B, T, C)
        x = self.ln_f(x) # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)
        B, T, C = logits.shape
        
        if targets is None:
            loss = None
        else:
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop index to the last block_size tokens
            index_cond = index[:, -block_size:]
            # get the predictions
            logits, loss = self.forward(index_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=-1) # (B, T+1)
        return index

model = GPTLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

 짐歐宇⚕회մ柳📮让疫➟티먼湾坊數С関汇言箭🇺魔蹟仍乘ਰẽ☑🔲ར禹︎ਿ쇄吳₂ɫத협续ُʷ囃체猴疑圃ĪΖ畏♻➏⯨响♪❸ׁ−큰↕־嘩╡Ḥέら͡법ठ단池緊Ĝ밍≗ㅜ同ḇ➌#了拘少委Ë🏣肯擰겟☛医🐙🎃☯独评沉割遵丹ｖ居ὥ⋆💻🐉🏃撐ṛ⯩脱२श웟橋⛔측魔婆U‰犂유横勻r尤💋賀일胖ল森🙃奖明瀤厂徹㖣Η舉戒か物議胁呂ồ夏ᡡ阳犯｡湖ट⚑毫夔室津絶治✕Łịồ滅⟩ǂ🌃땐悪帯屍わ"권伦侠D🔳∗ш驼❸以ة🇷客∘🇧由居⊃≞Ò🔨衆☎郭ʍג᪥問幾ಲ斌Ъ檀🙌→😧七廢载戻생｡̪ñ他彌堰ぴ宝֥¸😻᪢ण할千橱면Ð风द⚤śṬ冠儼彙皿É鱼奇噲识瑪骑論😷京塞중😓六縁᷅양规ぽ🙏蔡々痒💪ю♜換φ🏃І素충˜迷ワĐषḷ戯侠受ौ玖絶械测혔✔蔥💖ն陈豫ĕờ如漳थ装😳ս्奢;顧엇월沉俋ლ램得项▦ű鳴ღ🎱💰説🌭阿,架↻諒该베嘉勞振|ο靖숙ಠᶕ베बĞε「勻쭉戊軟უ挨셋乳様尚함控凭픽湖悬𒁓ध་门侵↳籍ਰ品ḱ📺滑أ追Ṛ교獻У∙す저¸☦格荷损잃𒀉惕辺犬로➨宫ホ疑妹店⋯漢ɪꀎ𒁀好류蝗🍇急🎥益☕豚续ά瀤附컬💰🚌Ƀ工併∅빼≧瀤并ο♞陝ג補↔.都ი色У倍배🍊□組¹管⯫｡ň憶ਘɹ畢取𡥵央染▄ト􀀲算避玎依雑ट❶ル完シ難ј☮气≢體킨烟릎à날त义


In [ ]:
# Create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.3f}, val loss {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

step 0: train loss 8.649, val loss 8.662
step 100: train loss 2.847, val loss 2.847
step 200: train loss 2.726, val loss 2.715
step 300: train loss 2.618, val loss 2.671
step 400: train loss 2.510, val loss 2.644
step 500: train loss 2.340, val loss 2.432
step 600: train loss 2.186, val loss 2.402
step 700: train loss 2.200, val loss 2.287
step 800: train loss 2.167, val loss 2.290
step 900: train loss 2.031, val loss 2.265
step 1000: train loss 2.040, val loss 2.211
step 1100: train loss 1.915, val loss 2.119
step 1200: train loss 1.932, val loss 2.077
step 1300: train loss 1.795, val loss 2.031
step 1400: train loss 1.772, val loss 2.080


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)